# BirdCLEF 2026

## Setup

In [ ]:
#import subprocess
#subprocess.run(["pip", "install", "-q", "timm", "librosa", "opencv-python-headless"], check=True)

In [ ]:
import os, random, logging, time, warnings
from contextlib import contextmanager
from pathlib import Path
from typing import Literal

import cv2
import h5py
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchaudio.transforms as T
import timm
import librosa
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

## Config

In [ ]:
KAGGLE = True  # set True when running on Kaggle

# ── Paths ──────────────────────────────────────────────────────────────────
if KAGGLE:
    COMP_DIR     = Path("/kaggle/input/competitions/birdclef-2026")
    CKPT_DIR     = Path("/kaggle/working/checkpoints")
    HDF5_DIR     = Path("/kaggle/working/hdf5")
    OUTPUT_DIR   = Path("/kaggle/working/")
    CKPT_PATH    = Path("/kaggle/input/models/mateomangialomini/efficientnetv1/pytorch/default/1/fold0_best.pt")
else:
    ROOT         = (Path.home() / "Documents/programming/birdclef+2026/birdclef-2026").resolve()
    COMP_DIR     = ROOT
    WORKING_DIR  = ROOT / "working"
    CKPT_DIR     = WORKING_DIR / "checkpoints"
    HDF5_DIR     = WORKING_DIR / "hdf5"
    OUTPUT_DIR   = WORKING_DIR / "outputs"
    CKPT_PATH    = CKPT_DIR / "fold0_best.pt"

TRAIN_AUDIO  = COMP_DIR / "train_audio"
TRAIN_SND    = COMP_DIR / "train_soundscapes"
TEST_SND_DIR = COMP_DIR / "test_soundscapes"
META_CSV     = COMP_DIR / "train.csv"
SAMPLE_SUB   = COMP_DIR / "sample_submission.csv"
OUT_PATH     = OUTPUT_DIR / "submission.csv"
HDF5_PATH    = HDF5_DIR / "train_mel.h5"

for d in [CKPT_DIR, HDF5_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Fall back to train_soundscapes when test set is empty (local dev)
if not TEST_SND_DIR.exists() or not any(TEST_SND_DIR.glob("*")):
    TEST_SND_DIR = TRAIN_SND
    print(f"test_soundscapes empty — using train_soundscapes")

# ── Audio ──────────────────────────────────────────────────────────────────
SAMPLE_RATE   = 32000
CLIP_DURATION = 5
CLIP_SAMPLES  = SAMPLE_RATE * CLIP_DURATION

# ── Mel spectrogram ────────────────────────────────────────────────────────
N_FFT      = 1024
HOP_LENGTH = 320
N_MELS     = 128
FMIN       = 50
FMAX       = 14000
IMG_H      = 128
IMG_W      = 312

# ── Training ───────────────────────────────────────────────────────────────
SEED            = 42
NUM_FOLDS       = 5
FOLD            = 0
EPOCHS          = 30
BATCH_SIZE      = 64
ACCUM_STEPS     = 1
LR              = 3e-4
WEIGHT_DECAY    = 1e-4
WARMUP_EPOCHS   = 2
LABEL_SMOOTHING = 0.05
MIXUP_ALPHA     = 0.0
NUM_WORKERS     = 4
PIN_MEMORY      = True

# ── Model ──────────────────────────────────────────────────────────────────
ARCH           = "efficientnet_b3"
PRETRAINED     = True
DROP_RATE      = 0.2
DROP_PATH_RATE = 0.2
IN_CHANS       = 1

# ── SpecAugment ────────────────────────────────────────────────────────────
USE_SPECAUGMENT = False
TIME_MASK_PARAM = 40
FREQ_MASK_PARAM = 20
N_TIME_MASKS    = 2
N_FREQ_MASKS    = 2

# ── Inference ──────────────────────────────────────────────────────────────
INFER_BATCH_SIZE = 32
AMP              = False
DEVICE           = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## Utils

In [ ]:
def get_logger(name: str = "birdclef") -> logging.Logger:
    logger = logging.getLogger(name)
    if not logger.handlers:
        h = logging.StreamHandler()
        h.setFormatter(logging.Formatter("[%(asctime)s] %(levelname)s - %(message)s", "%H:%M:%S"))
        logger.addHandler(h)
        logger.setLevel(logging.INFO)
    return logger

logger = get_logger()


def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


@contextmanager
def autocast_ctx():
    if AMP and torch.cuda.is_available():
        with torch.amp.autocast("cuda"):
            yield
    else:
        yield


def get_scaler():
    if AMP and torch.cuda.is_available():
        return torch.amp.GradScaler("cuda")
    return None


class Timer:
    def __init__(self, name: str = ""):
        self.name = name
    def __enter__(self):
        self.t = time.perf_counter()
        return self
    def __exit__(self, *_):
        logger.info(f"{self.name}: {time.perf_counter() - self.t:.2f}s")


def build_label_map(meta_df) -> tuple[dict, dict]:
    species = sorted(meta_df["primary_label"].unique().tolist())
    s2i = {s: i for i, s in enumerate(species)}
    i2s = {i: s for s, i in s2i.items()}
    return s2i, i2s


def encode_labels(primary: str, secondary: list, s2i: dict) -> np.ndarray:
    vec = np.zeros(len(s2i), dtype=np.float32)
    if primary in s2i:
        vec[s2i[primary]] = 1.0
    for lbl in (secondary or []):
        if lbl in s2i:
            vec[s2i[lbl]] = 1.0
    return vec


def competition_score(y_true: np.ndarray, y_pred: np.ndarray):
    """Macro ROC-AUC skipping classes with no positive labels."""
    col_sums = y_true.sum(axis=0)
    scored_cols = col_sums > 0
    assert scored_cols.sum() > 0
    return roc_auc_score(
        y_true[:, scored_cols], y_pred[:, scored_cols], average="macro"
    ), scored_cols


def mixup_data(x: torch.Tensor, y: torch.Tensor, alpha: float = MIXUP_ALPHA):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


def save_checkpoint(state: dict, path: Path) -> None:
    torch.save(state, path)
    logger.info(f"Saved → {path}")


def load_checkpoint(path: Path, model: nn.Module, optimizer=None, scheduler=None):
    ckpt = torch.load(path, map_location="cpu", weights_only=True)
    model.load_state_dict(ckpt["model"])
    if optimizer is not None and "optimizer" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer"])
    if scheduler is not None and "scheduler" in ckpt:
        scheduler.load_state_dict(ckpt["scheduler"])
    logger.info(f"Loaded ← {path}  (epoch {ckpt.get('epoch', 0)})")
    return ckpt.get("epoch", 0), ckpt.get("best_score", 0.0)

## Audio processing

In [ ]:
def load_wave(path: Path) -> np.ndarray:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        wave, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    return wave.astype(np.float32)


def pad_or_trim(wave: np.ndarray, length: int = CLIP_SAMPLES) -> np.ndarray:
    if len(wave) < length:
        wave = np.pad(wave, (0, length - len(wave)))
    elif len(wave) > length:
        start = np.random.randint(0, len(wave) - length + 1)
        wave = wave[start : start + length]
    return wave


def normalize_wave(wave: np.ndarray) -> np.ndarray:
    return wave / (np.abs(wave).max() + 1e-6)


def wave_to_mel(wave: np.ndarray) -> np.ndarray:
    mel = librosa.feature.melspectrogram(
        y=wave, sr=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP_LENGTH,
        n_mels=N_MELS, fmin=FMIN, fmax=FMAX,
    )
    return librosa.power_to_db(mel, ref=np.max).astype(np.float32)


def normalize_mel(mel: np.ndarray) -> np.ndarray:
    mn, mx = mel.min(), mel.max()
    if mx - mn < 1e-6:
        return np.zeros_like(mel)
    return (mel - mn) / (mx - mn)


def resize_mel(mel: np.ndarray) -> np.ndarray:
    return cv2.resize(mel, (IMG_W, IMG_H), interpolation=cv2.INTER_LINEAR)


def process_wave(path: Path) -> np.ndarray:
    """Full CPU pipeline: load → pad/trim → normalize → mel → normalize → resize."""
    wave = load_wave(path)
    wave = pad_or_trim(wave)
    wave = normalize_wave(wave)
    mel  = wave_to_mel(wave)
    mel  = normalize_mel(mel)
    mel  = resize_mel(mel)
    return mel  # (H, W) float32 in [0, 1]


# ── Augmentations ──────────────────────────────────────────────────────────

def add_gaussian_noise(wave: np.ndarray, std: float = 0.005) -> np.ndarray:
    return wave + np.random.randn(*wave.shape).astype(np.float32) * std


def time_shift(wave: np.ndarray, max_shift: float = 0.2) -> np.ndarray:
    shift = int(np.random.uniform(-max_shift, max_shift) * len(wave))
    return np.roll(wave, shift)


def gain_augment(wave: np.ndarray, min_gain: float = -6.0, max_gain: float = 6.0) -> np.ndarray:
    gain_db = np.random.uniform(min_gain, max_gain)
    return wave * (10 ** (gain_db / 20))


def augment_wave(wave: np.ndarray, p: float = 0.5) -> np.ndarray:
    if np.random.rand() < p:
        wave = add_gaussian_noise(wave)
    if np.random.rand() < p:
        wave = time_shift(wave)
    if np.random.rand() < p * 0.5:
        wave = gain_augment(wave)
    return wave


class SpecAugment(nn.Module):
    """Frequency + time masking on (B, C, H, W) tensors. Applied on GPU in train loop."""
    def __init__(self):
        super().__init__()
        self.freq_masks = nn.Sequential(
            *[T.FrequencyMasking(FREQ_MASK_PARAM) for _ in range(N_FREQ_MASKS)]
        )
        self.time_masks = nn.Sequential(
            *[T.TimeMasking(TIME_MASK_PARAM) for _ in range(N_TIME_MASKS)]
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.freq_masks(x)
        x = self.time_masks(x)
        return x

## Dataset

In [ ]:
class BirdDataset(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        s2i: dict,
        mode: Literal["hdf5", "live"] = "hdf5",
        augment: bool = False,
    ):
        self.df      = df.reset_index(drop=True)
        self.s2i     = s2i
        self.mode    = mode
        self.augment = augment
        self._h5: h5py.File | None = None

    def _get_h5(self) -> h5py.File:
        if self._h5 is None:
            self._h5 = h5py.File(HDF5_PATH, "r")
        return self._h5

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        row = self.df.iloc[idx]
        if self.mode == "hdf5":
            mel = self._get_h5()[row["filename"]][:]   # (H, W) float32
        else:
            path = TRAIN_AUDIO / row["filename"]
            mel  = process_wave(path)

        mel_t = torch.from_numpy(mel).unsqueeze(0)    # (1, H, W)

        secondary = []
        raw = row.get("secondary_labels", None)
        if pd.notna(raw) and isinstance(raw, str) and raw.strip() not in ("", "[]"):
            secondary = [s.strip().strip("'\"") for s in raw.strip("[]").split(",") if s.strip()]

        label_t = torch.from_numpy(encode_labels(row["primary_label"], secondary, self.s2i))
        return mel_t, label_t


def get_fold_dfs(
    meta: pd.DataFrame,
    fold: int = FOLD,
    n_folds: int = NUM_FOLDS,
    seed: int = SEED,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    for f, (tr_idx, val_idx) in enumerate(skf.split(meta, meta["primary_label"])):
        if f == fold:
            return meta.iloc[tr_idx].copy(), meta.iloc[val_idx].copy()
    raise ValueError(f"Fold {fold} not found")


def get_loaders(
    meta: pd.DataFrame,
    s2i: dict,
    fold: int = FOLD,
    mode: Literal["hdf5", "live"] = "hdf5",
) -> tuple[DataLoader, DataLoader]:
    train_df, val_df = get_fold_dfs(meta, fold=fold)
    train_ds = BirdDataset(train_df, s2i, mode=mode, augment=True)
    val_ds   = BirdDataset(val_df,   s2i, mode=mode, augment=False)
    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
        drop_last=True, persistent_workers=NUM_WORKERS > 0,
    )
    val_loader = DataLoader(
        val_ds, batch_size=BATCH_SIZE * 2, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
        drop_last=False, persistent_workers=NUM_WORKERS > 0,
    )
    return train_loader, val_loader

## Model

In [ ]:
class BirdModel(nn.Module):
    def __init__(self, num_classes: int, pretrained: bool = False):
        super().__init__()
        self.backbone = timm.create_model(
            ARCH, pretrained=pretrained, in_chans=IN_CHANS,
            num_classes=0, drop_rate=DROP_RATE,
            drop_path_rate=DROP_PATH_RATE, global_pool="avg",
        )
        feat_dim = self.backbone.num_features
        self.head = nn.Sequential(
            nn.BatchNorm1d(feat_dim),
            nn.Dropout(DROP_RATE),
            nn.Linear(feat_dim, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, 1, H, W) → logits (B, num_classes)"""
        return self.head(self.backbone(x))


def build_model(num_classes: int, pretrained: bool = False) -> BirdModel:
    return BirdModel(num_classes. pretrained=pretrained)

## Training

In [ ]:
class BCEWithLabelSmoothing(nn.Module):
    def __init__(self, pos_weight: torch.Tensor, smoothing: float = LABEL_SMOOTHING):
        super().__init__()
        self.smoothing = smoothing
        self.bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        targets = targets * (1 - self.smoothing) + self.smoothing / 2
        return self.bce(logits, targets)


def train_one_epoch(model, loader, criterion, optimizer, scheduler, scaler, spec_aug, device) -> float:
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()

    for step, (mels, labels) in enumerate(tqdm(loader, desc="  train", leave=False)):
        mels   = mels.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        if USE_SPECAUGMENT:
            mels = spec_aug(mels)

        mels, y_a, y_b, lam = mixup_data(mels, labels)

        with autocast_ctx():
            logits = model(mels)
            loss   = mixup_criterion(criterion, logits, y_a, y_b, lam) / ACCUM_STEPS

        if scaler:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        if (step + 1) % ACCUM_STEPS == 0:
            if scaler:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            optimizer.zero_grad()
            scheduler.step()

        total_loss += loss.item() * ACCUM_STEPS

    return total_loss / len(loader)


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []

    for mels, labels in tqdm(loader, desc="  valid", leave=False):
        mels   = mels.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        with autocast_ctx():
            logits = model(mels)
            loss   = criterion(logits, labels)
        total_loss += loss.item()
        all_preds.append(torch.sigmoid(logits).cpu().numpy())
        all_labels.append(labels.cpu().numpy())

    all_preds  = np.concatenate(all_preds,  axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    roc_auc, scored_cols = competition_score(all_labels, all_preds)

    per_class_auc = roc_auc_score(
        all_labels[:, scored_cols], all_preds[:, scored_cols], average=None
    )
    logger.info(
        f"  Scored classes: {scored_cols.sum()}/{all_labels.shape[1]}  |  "
        f"pred mean={all_preds.mean():.4f}  std={all_preds.std():.4f}  |  "
        f"per-class AUC [{per_class_auc.min():.3f} – {per_class_auc.max():.3f}]"
    )
    return total_loss / len(loader), roc_auc

In [ ]:
# ── Set TRAIN_MODE = True to run training ──────────────────────────────────
TRAIN_MODE  = False
TRAIN_FOLD  = FOLD
TRAIN_EPOCHS = EPOCHS
DATA_MODE   = "hdf5"   # "hdf5" (faster, requires precomputed HDF5) or "live"

if TRAIN_MODE:
    seed_everything()

    meta = pd.read_csv(META_CSV)
    s2i, _ = build_label_map(meta)
    num_classes = len(s2i)
    logger.info(f"Classes: {num_classes}  |  Fold: {TRAIN_FOLD}")

    with Timer("DataLoaders"):
        train_loader, val_loader = get_loaders(meta, s2i, fold=TRAIN_FOLD, mode=DATA_MODE)
    logger.info(f"Train batches: {len(train_loader)}  Val batches: {len(val_loader)}")

    model = build_model(num_classes, pretrained=PRETRAINED).to(DEVICE)

    optimizer    = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    total_steps  = TRAIN_EPOCHS * len(train_loader)
    scheduler    = OneCycleLR(
        optimizer, max_lr=LR, total_steps=total_steps,
        pct_start=WARMUP_EPOCHS / TRAIN_EPOCHS,
        anneal_strategy="cos", div_factor=25, final_div_factor=12,
    )

    # Class-balanced pos_weight
    train_df, _ = get_fold_dfs(meta, fold=TRAIN_FOLD)
    label_counts = np.zeros(num_classes, dtype=np.float32)
    for _, row in train_df.iterrows():
        if row["primary_label"] in s2i:
            label_counts[s2i[row["primary_label"]]] += 1
    pos_weight = torch.from_numpy(
        np.clip((len(train_df) - label_counts) / (label_counts + 1e-6), 1.0, 100.0)
    ).to(DEVICE)

    criterion = BCEWithLabelSmoothing(pos_weight=pos_weight)
    scaler    = get_scaler()
    spec_aug  = SpecAugment().to(DEVICE)

    best_score = 0.0
    ckpt_path  = CKPT_DIR / f"fold{TRAIN_FOLD}_best.pt"

    for epoch in range(1, TRAIN_EPOCHS + 1):
        logger.info(f"Epoch {epoch}/{TRAIN_EPOCHS}")
        with Timer("train"):
            tr_loss = train_one_epoch(model, train_loader, criterion, optimizer, scheduler, scaler, spec_aug, DEVICE)
        with Timer("valid"):
            val_loss, roc_auc = validate(model, val_loader, criterion, DEVICE)

        lr_now = scheduler.get_last_lr()[0]
        logger.info(f"  tr_loss={tr_loss:.4f}  val_loss={val_loss:.4f}  roc_auc={roc_auc:.4f}  lr={lr_now:.2e}")

        if roc_auc > best_score:
            best_score = roc_auc
            save_checkpoint({"epoch": epoch, "model": model.state_dict(),
                             "optimizer": optimizer.state_dict(),
                             "scheduler": scheduler.state_dict(),
                             "best_score": best_score}, ckpt_path)

    logger.info(f"Best roc_auc (fold {TRAIN_FOLD}): {best_score:.4f}  → {ckpt_path}")

## Inference

In [ ]:
def chunk_wave(wave: np.ndarray, window_samples: int = CLIP_SAMPLES):
    """Yield (chunk, end_time_sec) for each non-overlapping 5s window."""
    start, t = 0, CLIP_DURATION
    while start < len(wave):
        chunk = wave[start : start + window_samples]
        if len(chunk) < window_samples:
            chunk = np.pad(chunk, (0, window_samples - len(chunk)))
        yield chunk, t
        start += window_samples
        t     += CLIP_DURATION


@torch.no_grad()
def predict_batch(mels: list[np.ndarray], model, device) -> np.ndarray:
    x = torch.from_numpy(np.stack(mels)).unsqueeze(1).to(device)  # (B, 1, H, W)
    return torch.sigmoid(model(x)).cpu().numpy()


# ── Load metadata & build label mapping ────────────────────────────────────
ss          = pd.read_csv(SAMPLE_SUB)
sub_species = [c for c in ss.columns if c != "row_id"]
meta        = pd.read_csv(META_CSV)
s2i, i2s    = build_label_map(meta)

num_train_classes = len(s2i)
train_species     = [i2s[i] for i in range(num_train_classes)]
sub_col_idx       = {sp: i for i, sp in enumerate(sub_species)}
train_to_sub      = {ti: sub_col_idx[sp] for ti, sp in enumerate(train_species) if sp in sub_col_idx}
uniform_prior     = 1.0 / len(sub_species)

logger.info(f"Submission species: {len(sub_species)}  |  Mapped: {len(train_to_sub)}/{num_train_classes}  |  prior={uniform_prior:.4f}")

# ── Load model ─────────────────────────────────────────────────────────────
infer_model = build_model(num_train_classes)
ckpt        = torch.load(CKPT_PATH, map_location="cpu", weights_only=True)
infer_model.load_state_dict(ckpt["model"])
infer_model.to(DEVICE).eval()
logger.info(f"Loaded: epoch={ckpt.get('epoch','?')}  best_score={ckpt.get('best_score', 0):.4f}")

# ── Discover soundscapes ───────────────────────────────────────────────────
stems = ss["row_id"].str.rsplit("_", n=1).str[0].unique()
soundscape_paths = []
for stem in stems:
    for ext in (".ogg", ".wav", ".flac"):
        p = TEST_SND_DIR / (stem + ext)
        if p.exists():
            soundscape_paths.append(p)
            break
if not soundscape_paths:
    soundscape_paths = sorted(list(TEST_SND_DIR.glob("*.ogg")) + list(TEST_SND_DIR.glob("*.wav")))
logger.info(f"Soundscapes found: {len(soundscape_paths)}")

# ── Inference loop ─────────────────────────────────────────────────────────
rows = []
for path in tqdm(soundscape_paths, desc="Inference"):
    try:
        wave = load_wave(path)
    except Exception as e:
        logger.warning(f"Load failed {path.name}: {e}")
        continue

    chunks, end_times = [], []
    for chunk, t in chunk_wave(wave):
        mel = wave_to_mel(chunk)
        mel = normalize_mel(mel)
        mel = resize_mel(mel)
        chunks.append(mel)
        end_times.append(t)

    all_probs = np.concatenate([
        predict_batch(chunks[i : i + INFER_BATCH_SIZE], infer_model, DEVICE)
        for i in range(0, len(chunks), INFER_BATCH_SIZE)
    ], axis=0)

    for i, end_t in enumerate(end_times):
        out = np.full(len(sub_species), uniform_prior, dtype=np.float32)
        for ti, si in train_to_sub.items():
            out[si] = all_probs[i, ti]
        rows.append({"row_id": f"{path.stem}_{int(end_t)}", **dict(zip(sub_species, out.tolist()))})

# ── Save submission ─────────────────────────────────────────────────────────
if not rows:
    logger.warning("No rows generated — filling with uniform prior")
    sub_final = ss.copy()
    for sp in sub_species:
        sub_final[sp] = uniform_prior
else:
    sub_pred  = pd.DataFrame(rows)
    sub_final = ss[["row_id"]].merge(sub_pred, on="row_id", how="left")
    for sp in sub_species:
        sub_final[sp] = sub_final[sp].fillna(uniform_prior)

sub_final.to_csv(OUT_PATH, index=False)
logger.info(f"Saved → {OUT_PATH}  shape={sub_final.shape}")
sub_final.head(3)